SOW-BKI230A Deep Learning<br>Spring 2025

Assignment 3<br>Recurrent Neural Networks

**Name**:

Angelina Podoļako

**S-number**:

s1125886

### Generating Song Lyrics with Recurrent Neural Networks:

The goal of this assignment is to help you understand the code provided and to experiment with it in order to generate the best song lyrics that you can. By the end of the assignment, you should have a basic understanding of how recurrent neural networks can be used to generate text and should be able to experiment with different hyperparameters and seed texts to generate different styles of song lyrics.

Tasks:

1. Begin by reading through the code provided and make sure you understand what each line of code does. If you have any questions, feel free to ask your TA or use external resources to help you.

2. Once you understand the code, run it and take a look at the output. The code will generate a random sequence of words based on the seed text provided. The quality of the output will depend on the hyperparameters used and the seed text. Experiment with different hyperparameters and seed texts to see how they affect the output.

3. Modify the hyperparameters provided in the code (`BATCH_SIZE`, `DRAW_NUM`, `DROPOUT_PROB`, `EMBEDDING_SIZE`, `EPOCH_NUM`, `HIDDEN_SIZE`, `LAYER_NUM`, `LEARNING_RATE`, `LOGIT_TEMP`, `SEQ_SIZE`, `VOCAB_SIZE`) and see how they affect the output. For example, you could try increasing the hidden size or the number of layers to see if that improves the quality of the output.

4. Try using different seed texts to generate different styles of song lyrics. You could use lyrics from your favorite songs, or you could try using random sentences or phrases. See how the output changes depending on the seed text used.

5. Once you have experimented with different hyperparameters and seed texts, try generating the best song lyrics that you can. You can do this by running the code multiple times with different hyperparameters and seed texts until you find a sequence of words that you like. You can also try modifying the code to improve the quality of the output. For example, you could try using a different neural network architecture or a different training algorithm.

6. Write a **short report** describing your experiments and the best song lyrics that you were able to generate. Your report should include a description of the hyperparameters and seed texts used, as well as any modifications you made to the code. It should also include a sample of the song lyrics that you generated. Include your report at the end of this notebook.

### Cell 1:

This cell imports necessary Python libraries and modules. The libraries imported are:

1. `collections`: This library provides high-performance container datatypes, such as Counter.
2. `math`: This library provides mathematical functions like `exp` for calculating perplexity.
3. `numpy`: This library provides support for working with arrays and various numerical operations.
4. `pickle`: This library is used for serializing and deserializing Python objects.
5. `random`: This library contains various functions for generating random numbers.
6. `torch`: This is the PyTorch library, which provides tensor computation and deep learning functionalities.
7. `torch.nn`: This is the neural network module of PyTorch.
8. `torch.utils.data`: This module contains useful classes like DataLoader and Dataset for handling data in PyTorch.

In [ ]:
import collections
import math
import numpy as np
import pickle
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

### Cell 2:

This cell defines various hyperparameters, seeds the random number generators for reproducibility, and sets up the device for computation (GPU if available, otherwise CPU). The defined hyperparameters are:

1. `BATCH_SIZE`: The number of samples in a mini-batch for training.
2. `DRAW_NUM`: The number of words to generate when drawing a sample.
3. `DROPOUT_PROB`: The probability of dropout in the GRU layers.
4. `EMBEDDING_SIZE`: The size of the word embeddings.
5. `EPOCH_NUM`: The number of training epochs.
6. `HIDDEN_SIZE`: The size of the hidden state in the GRU layers.
7. `LAYER_NUM`: The number of GRU layers.
8. `LEARNING_RATE`: The learning rate for the optimizer.
9. `LOGIT_TEMP`: The temperature for controlling the randomness in the generated samples.
10. `MODEL_NAME`: The name of the saved model.
11. `SEED_TEXT`: The seed text for generating a sample.
12. `SEQ_SIZE`: The length of input sequences.
13. `VOCAB_SIZE`: The size of the vocabulary.

The random number generators for `random`, `numpy`, and `torch` are seeded with the same value (42) to ensure consistent results across runs.

In [ ]:
BATCH_SIZE     = 128
DRAW_NUM       = 64
DROPOUT_PROB   = 0.5
EMBEDDING_SIZE = 64
EPOCH_NUM      = 10
HIDDEN_SIZE    = 64
LAYER_NUM      = 2
LEARNING_RATE  = 1e-3
LOGIT_TEMP     = 0.7
MODEL_NAME     = "model"
SEED_TEXT      = "Sweet dreams are made of this Who am I to disagree"
SEQ_SIZE       = 64
VOCAB_SIZE     = 10000

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#!python -m textblob.download_corpora

### Cell 3:

This cell downloads and unzips the dataset containing song lyrics. It then loads the dataset using the `pickle` library and assigns it to the variable `seqs`.

In [ ]:
# !pip install contractions

# import contractions
# import glob
# import nltk
# import textblob

# nltk.download("punkt")

# !wget -nc https://umguec.github.io/file-sharing/kaggle_song_lyrics_dataset.zip
# !unzip -n kaggle_song_lyrics_dataset.zip -d kaggle_song_lyrics_dataset

# seqs = []
# for pathname in sorted(glob.glob("kaggle_song_lyrics_dataset/*.txt")):
#     with open(pathname) as f:
#         seqs += textblob.TextBlob(contractions.fix(f.read().lower())).correct().words

!wget -nc https://umguec.github.io/file-sharing/kaggle_song_lyrics_dataset.pkl.zip
!unzip -n kaggle_song_lyrics_dataset.pkl.zip -d kaggle_song_lyrics_dataset

with open("kaggle_song_lyrics_dataset/kaggle_song_lyrics_dataset.pkl", "rb") as f:
    seqs = pickle.load(f)

--2025-04-09 11:46:39--  https://umguec.github.io/file-sharing/kaggle_song_lyrics_dataset.pkl.zip
Resolving umguec.github.io (umguec.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to umguec.github.io (umguec.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4023902 (3.8M) [application/zip]
Saving to: ‘kaggle_song_lyrics_dataset.pkl.zip’

kaggle_song_lyrics_ 100%[===================>]   3.84M  --.-KB/s    in 0.02s   

2025-04-09 11:46:39 (155 MB/s) - ‘kaggle_song_lyrics_dataset.pkl.zip’ saved [4023902/4023902]

Archive:  kaggle_song_lyrics_dataset.pkl.zip
  inflating: kaggle_song_lyrics_dataset/kaggle_song_lyrics_dataset.pkl  
  inflating: kaggle_song_lyrics_dataset/__MACOSX/._kaggle_song_lyrics_dataset.pkl  


### Cell 4:

This cell creates a vocabulary from the dataset and maps words to indices and indices to words. It first calculates the frequency of each word using `collections.Counter` and selects the top `VOCAB_SIZE` words, adding the `<unk>` token for unknown words. Then, it creates two dictionaries, `idx_to_word` and `word_to_idx`, to map indices to words and vice versa.

In [ ]:
vocab       = ["<unk>"] + (lambda counter: sorted(counter, key=counter.get, reverse=True))(collections.Counter(seqs))[:VOCAB_SIZE - 1]
idx_to_word = {idx: word for idx, word in enumerate(vocab)}
word_to_idx = {word: idx for idx, word in enumerate(vocab)}

### Cell 5:

This cell defines a custom PyTorch Dataset class, `SeqDataset`, that takes in the device, sequence size, and sequences as input. It has two main methods:

1. `__len__()`: This method returns the length of the dataset, which is the number of sequences minus the sequence size minus one.
2. `__getitem__(self, idx)`: This method returns the input sequence and target sequence for a given index by slicing the sequences list and converting it into a tensor.

The training, validation, and testing datasets are then created using this custom class, and their respective DataLoader instances are created for easier data handling during training and evaluation.

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, device, seq_size, seqs):
        super(SeqDataset, self).__init__()
        self.device   = device
        self.seq_size = seq_size
        self.seqs     = seqs

    def __len__(self):
        return len(self.seqs) - self.seq_size - 1

    def __getitem__(self, idx):
        in_seq     = torch.tensor(self.seqs[idx    :idx + self.seq_size    ], dtype=torch.long, device=self.device)
        target_seq = torch.tensor(self.seqs[idx + 1:idx + self.seq_size + 1], dtype=torch.long, device=self.device)
        return in_seq, target_seq

train_set = SeqDataset(device, SEQ_SIZE, [word_to_idx.get(word, 0) for word in seqs[                        :int(0.8 * len(seqs))]])
val_set   = SeqDataset(device, SEQ_SIZE, [word_to_idx.get(word, 0) for word in seqs[int(0.8 * len(seqs)) + 1:int(0.9 * len(seqs))]])
test_set  = SeqDataset(device, SEQ_SIZE, [word_to_idx.get(word, 0) for word in seqs[int(0.9 * len(seqs)) + 1:                    ]])

train_loader = DataLoader(train_set, BATCH_SIZE, True )
val_loader   = DataLoader(val_set  , BATCH_SIZE, False)
test_loader  = DataLoader(test_set , BATCH_SIZE, False)

### Cell 6:

This cell defines the `Model` class, which is a subclass of `nn.Module`. It has the following layers:

1. `embedding`: An Embedding layer to convert word indices into word embeddings.
2. `gru`: A GRU layer with the specified number of layers, hidden size, and dropout probability.
3. `linear`: A linear layer that maps the hidden state to the vocabulary size for output probabilities.

The `Model` class has the following methods:

1. `forward(self, in_sequence, hidden_state=None)`: This method computes the forward pass given an input sequence and an optional hidden state. It computes the embedded sequence, the hidden sequence, and the output sequence using the defined layers.
2. `draw(self, in_sequence, logit_temp=1.0)`: This method generates a random sample given an input sequence and a logit temperature. It computes the output sequence, applies a softmax function with the temperature, and samples from the resulting probability distribution.

In [ ]:
class Model(nn.Module):
    def __init__(self, dropout_prob, embedding_size, hidden_size, layer_num, vocab_size):
        super(Model, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.gru       = nn.GRU(embedding_size, hidden_size, layer_num, batch_first=True, dropout=dropout_prob)
        self.linear    = nn.Linear(hidden_size, vocab_size)

    def forward(self, in_sequence, hidden_state=None):
        embedding_seq            = self.embedding(in_sequence)
        hidden_seq, hidden_state = self.gru(embedding_seq, hidden_state)
        out_seq                  = self.linear(hidden_seq)
        return out_seq, hidden_state

    def draw(self, in_sequence, logit_temp=1.0):
        out_seq, _  = self(in_sequence)
        prob_dist   = torch.softmax(out_seq[0, -1] / logit_temp, 0)
        rand_sample = torch.multinomial(prob_dist, 1).item()
        return rand_sample

### Cell 7:

This cell initializes the model, criterion (CrossEntropyLoss), and optimizer (Adam) with the specified hyperparameters. The model is moved to the device (GPU or CPU) for computation.

In [ ]:
model     = Model(DROPOUT_PROB, EMBEDDING_SIZE, HIDDEN_SIZE, LAYER_NUM, VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), LEARNING_RATE)

### Cell 8:

This cell trains the model for the specified number of epochs. For each epoch, it performs the following steps:

1. Sets the model to training mode.
2. Initializes the training loss to 0.
3. Iterates over the training data using the `train_loader`.
4. Computes the output sequence and loss for each input and target sequence.
5. Updates the training loss and performs backpropagation.
6. Computes the average training loss and perplexity.
7. Prints the training loss and perplexity for the current epoch.
8. Sets the model to evaluation mode.
9. Initializes the validation loss to 0.
10. Iterates over the validation data using the `val_loader` without computing gradients.
11. Computes the output sequence and loss for each input and target sequence.
12. Updates the validation loss.
13. Computes the average validation loss and perplexity.
14. Prints the validation loss and perplexity for the current epoch.
15. If the validation loss is lower than the previous minimum loss, saves the model's state dictionary.

This process continues until all epochs are completed.

In [ ]:
min_loss = float("inf")

for epoch in range(EPOCH_NUM):
    print(f"Epoch: {epoch + 1}/{EPOCH_NUM}")

    model.train()
    train_loss = 0.0

    for in_seq, target_seq in train_loader:
        out_seq, _  = model(in_seq)
        loss        = criterion(out_seq.view(-1, VOCAB_SIZE), target_seq.view(-1))
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss       /= len(train_loader)
    train_perplexity  = math.exp(train_loss)
    print(f"Train loss: {train_loss:.4f}, Train perplexity: {train_perplexity:.4f}")

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for in_seq, target_seq in val_loader:
            out_seq, _  = model(in_seq)
            loss        = criterion(out_seq.view(-1, VOCAB_SIZE), target_seq.view(-1))
            val_loss   += loss.item()

    val_loss       /= len(val_loader)
    val_perplexity  = math.exp(val_loss)
    print(f"Val loss: {val_loss:.4f}, Val perplexity: {val_perplexity:.4f}")

    if val_loss < min_loss:
        min_loss = val_loss
        torch.save(model.state_dict(), f"kaggle_song_lyrics_dataset/{MODEL_NAME:s}.pt")

Epoch: 1/10


### Cell 9:

This cell evaluates the model on the test set by performing the following steps:

1. Loads the saved state dictionary of the model.
2. Sets the model to evaluation mode.
3. Initializes the test loss to 0.
4. Iterates over the test data using the `test_loader` without computing gradients.
5. Computes the output sequence and loss for each input and target sequence.
6. Updates the test loss.
7. Computes the average test loss and perplexity.
8. Prints the test loss and perplexity.

In [ ]:
model = Model(DROPOUT_PROB, EMBEDDING_SIZE, HIDDEN_SIZE, LAYER_NUM, VOCAB_SIZE).to(device)
model.load_state_dict(torch.load(f"kaggle_song_lyrics_dataset/{MODEL_NAME:s}.pt"))

model.eval()
test_loss = 0.0

with torch.no_grad():
    for in_seq, target_seq in test_loader:
        out_seq, _  = model(in_seq)
        loss        = criterion(out_seq.view(-1, VOCAB_SIZE), target_seq.view(-1))
        test_loss  += loss.item()

test_loss       /= len(test_loader)
test_perplexity  = math.exp(test_loss)
print(f"Test loss: {test_loss:.4f}, Test perplexity: {test_perplexity:.4f}")

Test loss: 5.2344, Test perplexity: 187.6127


### Cell 10:

This code block generates a random text sample using the trained model:

1. Loads the saved state dictionary of the model.
2. Sets the model to evaluation mode.
3. Initializes the `rand_samples` list with the indices of the words in the seed text.
4. Iterates `DRAW_NUM` times and performs the following steps:
   - Converts the last `SEQ_SIZE` words in the `rand_samples` list into a tensor.
   - Generates a random word index using the `draw` method of the model and appends it to the `rand_samples` list.
5. Converts the `rand_samples` list into a text string by mapping the indices to words using the `idx_to_word` dictionary.
6. Prints the generated text.

In [ ]:
model = Model(DROPOUT_PROB, EMBEDDING_SIZE, HIDDEN_SIZE, LAYER_NUM, VOCAB_SIZE).to(device)
model.load_state_dict(torch.load(f"kaggle_song_lyrics_dataset/{MODEL_NAME:s}.pt"))

model.eval()
rand_samples = [word_to_idx.get(word, 0) for word in SEED_TEXT.lower().split()]

with torch.no_grad():
    for _ in range(DRAW_NUM):
        in_sequence = torch.tensor([rand_samples[-SEQ_SIZE:]], dtype=torch.long, device=device)
        rand_samples.append(model.draw(in_sequence))

rand_text = " ".join(idx_to_word.get(idx, "<unk>") for idx in rand_samples)
print(rand_text)

sweet dreams are made of this who am i to disagree <unk> me yes i am calling my space short here in the head fall and made to love those days of the door back start to be exactly please good with me my sight because i do not fuck no longer is stomach inside the way kill there make us think you better have something no no more money and that winning and pitt



---

---


# Raport

## Default settings

After proper exploring the code and deep learning of all parameteres the model has, I have runned the code with the starting hyperparametres without changing anything.   


```
Default settings:  
BATCH_SIZE     = 32  
DRAW_NUM       = 32  
DROPOUT_PROB   = 0.5  
EMBEDDING_SIZE = 32  
EPOCH_NUM      = 10  
HIDDEN_SIZE    = 32  
LAYER_NUM      = 2  
LEARNING_RATE  = 1e-3  
LOGIT_TEMP     = 1.0  
MODEL_NAME     = "model"  
SEED_TEXT      = "The Answer to the Ultimate Question of Life the Universe and Everything is"  
SEQ_SIZE       = 32  
VOCAB_SIZE     = 10000  
  
Default seed text:  
The Answer to the Ultimate Question of Life the Universe and Everything is  
```
What I got:  
**the answer to the ultimate question of life the universe and everything is** *a beautiful time i will do more with short here get running on fall and do not love to you having the money back start blind he smokes the good girls fading*


## Learning parameteres:

I got this line above, consisting of 32 new words. It has some meaning, but I did not like it so much. Also training of the model tooked about 35 minutes. So, I started learn how each of hyperparameters affect the code, to change them in manner I would like.   

Of course I could try to run code a few times with changing only one hyperparameter per run. But then if I need to change each parameter separately (one run with higher value per parameter, another with lower value) to got new text, that meant I need to do  
 11(values to change) * 2(one time lower, another higher) = 22 runs * 50 minutes(how much time it takes in best case(google collab asks to pay money)). =1110 minutes = 18 hours.   

 Since I do not have so much time, I had learned with the internet and ChatGPT(which is allowed to use during this course) help.  
   
     
  
---
  
Firs of all, what I learned about parameters:


### Comparison of Hyperparameter Effects

| Hyperparameter (def val)  | Higher Value – Effect                                          | Example (High) | Lower Value – Effect                                         | Example (Low) |
|------------------|---------------------------------------------------------------|----------------|---------------------------------------------------------------|----------------|
| `BATCH_SIZE  (32) `  | Faster training, less stable gradients                        | `64`           | Slower training, more stable updates                          | `16`           |
| `DRAW_NUM     (32) ` | More output tokens generated, more variety                    | `64`           | Fewer tokens, shorter or less diverse generation              | `8`            |
| `DROPOUT_PROB  (0.5)` | Stronger regularization, prevents overfitting                 | `0.7`          | Weaker regularization, risk of overfitting                    | `0.1`          |
| `EMBEDDING_SIZE(32)` | Richer word representations, more expressive model            | `128`          | Less expressive, fewer features captured                      | `16`           |
| `EPOCH_NUM   (10)`   | More learning opportunities, risk of overfitting              | `50`           | May underfit the data                                         | `5`            |
| `HIDDEN_SIZE (32)`   | Better modeling of long-term dependencies                     | `128`          | Less capacity for complex patterns                            | `16`           |
| `LAYER_NUM    (2)`  | Deeper model, learns complex structures                       | `3`            | Simpler model, faster but may underperform                    | `1`            |
| `LEARNING_RATE (1e-3)` | Faster convergence, risk of instability                       | `1e-2`         | Slower, stable learning                                        | `1e-4`         |
| `LOGIT_TEMP  (1.0)`   | More random and creative outputs                              | `1.5`          | More deterministic, repetitive outputs                        | `0.7`          |
| `SEQ_SIZE   (32) `   | Considers longer context when generating or training          | `64`           | Shorter memory, may miss dependencies                         | `16`           |
| `VOCAB_SIZE  (10000)`   | Larger vocabulary, captures more diverse language             | `20000`        | Smaller vocabulary, more `<unk>` tokens                       | `5000`         |


## Changing the hyperparameters of the model

I changes some of parameters. For example now I have lower BATCH_SIZE, DRAW_NUM, EMBEDDING_SIZE, LOGIT_TEMP and SEQ_SIZE. But also I made higher DROPUT_PROB, HIDDEN_SIZE, LAYER_NUM, LEARNING_RATE. EPOCH_NUM and VOCAB_SIZE has been unchanged.
```
BATCH_SIZE     = 25
DRAW_NUM       = 25
DROPOUT_PROB   = 0.7
EMBEDDING_SIZE = 25
EPOCH_NUM      = 10
HIDDEN_SIZE    = 64
LAYER_NUM      = 3
LEARNING_RATE  = 1e-5
LOGIT_TEMP     = 0.5
MODEL_NAME     = "model"
SEED_TEXT      = "The Answer to the Ultimate Question of Life the Universe and Everything is"
SEQ_SIZE       = 25
VOCAB_SIZE     = 10000

```



I got new line:
**the answer to the ultimate question of life the universe and everything is** *a world is i shared me my lost short here how running on crazy and made out maybe those time of club and back start*

To be honest, I liked the text more, I don't know why. despite the fact that it is quite possible to comprehend it, it is more like a pop rap, which does not carry a heavy semantic load.

## Trying to find parameters I want to use for new SEED_TEXT

After a few runs with different parameters and checking result with the same SEED_TEXT, I decided to use these ones:



```
BATCH_SIZE     = 128
DRAW_NUM       = 64
DROPOUT_PROB   = 0.5
EMBEDDING_SIZE = 64
EPOCH_NUM      = 10
HIDDEN_SIZE    = 64
LAYER_NUM      = 2
LEARNING_RATE  = 1e-3
LOGIT_TEMP     = 0.7
MODEL_NAME     = "model"
SEED_TEXT      = "Sweet dreams are made of this Who am I to disagree"
SEQ_SIZE       = 64
VOCAB_SIZE     = 10000

```
To generate a meaningful and stylistically coherent continuation of the song lyrics   "*Sweet dreams are made of this / Who am I to disagree?*"   while maintaining reasonable training time. The selected hyperparameters optimize the trade-off between text quality and computational efficiency.



    BATCH_SIZE = 128 (Default: 32)
* Impact: Larger batches speed up training by processing more sequences in parallel.  
* Trade-off: Slightly reduces gradient precision but improves training stability.


    DRAW_NUM = 64 (Default: 32)
* Impact: Generates longer continuations (64 words instead of 32), allowing for more elaborate lyrical structures.



    DROPOUT_PROB = 0.5 (Default: 0.5)
* Impact:Maintains standard dropout rate to prevent overfitting.




    EMBEDDING_SIZE = 64 (Default: 32)

* Impact: Larger embeddings capture richer semantic relationships between words, improving lyrical expressiveness.



    EPOCH_NUM = 15 (Default: 10)
* Impact: Standard number of epochs balances learning and overfitting risks.



    HIDDEN_SIZE = 64 (Default: 32)
* Impact: A larger hidden state enables the model to remember longer lyrical sequences, improving flow and thematic consistency.



    LAYER_NUM = 2 (Default: 2)
* Impact: Keeps training efficient while allowing sufficient depth for learning song structure.



    LEARNING_RATE = 1e-2 (Default: 1e-3)
* Impact: A higher learning rate speeds up convergence, reducing training time.
* Trade-off: Risk of unstable training, but early stopping (via validation loss) prevents divergence.



    LOGIT_TEMP = 0.7 (Default: 1.0)
* Impact: Lower temperature reduces randomness, producing more predictable and lyrical outputs.



    SEQ_SIZE = 64 (Default: 32)
* Impact: Longer input sequences help the model better understand song structure and maintain context.



    VOCAB_SIZE = 10000 (Default: 10000)
* Impact: Maintains a balance between vocabulary coverage and computational efficiency.



Expected Results:
1. Stable Training: Conservative LEARNING_RATE with proper regularization.
2. Balanced Creativity: LOGIT_TEMP=0.7 yields poetic yet coherent lyrics.
3. Efficient Runtime: Optimized HIDDEN_SIZE and BATCH_SIZE for quicker training.
4. Reduced Overfitting: Higher DROPOUT_PROB and smaller HIDDEN_SIZE combat overfitting.
5. Longer, Structured Outputs: Enabled by DRAW_NUM and SEQ_SIZE.

I got:  
 **sweet dreams are made of this who am i to disagree **   *<unk> me yes i am calling my space short here in the head fall and made to love those days of the door back start to be exactly please good with me my sight because i do not fuck no longer is stomach inside the way kill there make us think you better have something no no more money and that winning and pitt*



I liked the result, mostly because it's a funny pop rap. But I would like to draw attention to the fact that due to the length of the answer, it is difficult to interpret it (there are no punctuation marks), as well as it is unclear why an unknown word designated as "unk" was used.  
In general, I saw a good result, but I know that there are problems that can be improved by changing the parameters. It is especially important to prevent overfitting like mine (I am aware of the problem, but due to the requirements, I will buy the paid Google collage version, I cannot afford it, I am already glad that my code has run).  


The selected parameters provide a balance between the speed of learning and the quality of the text. However, there is a risk of overfitting, which can be improved in the future by selecting better examples (which I cannot do at the moment). It is possible to implement an early stop with an increase in validation losses (which I have, but there is no way to fix at the moment).

In the future, it is possible to further test the balance between parameters, experiment with strings, and look for ways to get a quick response without retraining the model.

In conclusion, the current configuration is suitable for generating meaningful song sequels, but requires improvements to minimize overfitting. Further optimizations may include regularization, fine-tuning the temperature, and testing alternative architectures.